# Avaliacao Final: Classificador Hibrido MRI Otimizado

Este notebook utiliza os hiperparametros encontrados pelo **Optuna** para realizar o treinamento final e a inferencia em um conjunto de teste cego (Blind Test). Todos os resultados deste notebook serao salvos na subpasta `notebook/` dentro do diretorio de dados.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import json
import torch
import numpy as np

# Adiciona o diretorio raiz ao path
sys.path.append(os.path.abspath('../../../'))

from qml.group_works.group_01.loaders.mri_loader import get_dataloaders
from qml.group_works.group_01.models.hybrid_resnet import HybridResNet18
from qml.group_works.group_01.models.classical_resnet import ClassicalResNet18
from qml.group_works.group_01.trainer.training_loop import train_model, test_model
from qml.group_works.group_01.evaluation.metrics import plot_comparison, plot_confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# Define pasta de saida para o notebook
output_notebook_dir = '../../../data/group_works/group_01/notebook/'
os.makedirs(output_notebook_dir, exist_ok=True)

## 1. Carregamento dos Dados (Divisao 70/15/15)

In [ ]:
# Agora recebemos 3 loaders: Treino, Validacao e Teste (Cego)
train_loader, val_loader, test_loader = get_dataloaders(batch_size=8)
print(f'Treino: {len(train_loader.dataset)} amostras')
print(f'Validacao: {len(val_loader.dataset)} amostras')
print(f'Teste Cego: {len(test_loader.dataset)} amostras')

## 2. Carregar Melhores Hiperparametros
Lendo os arquivos JSON gerados pelo script de otimizacao.

In [ ]:
data_path = '../../../data/group_works/group_01/'

def load_best_params(filename):
    path = os.path.join(data_path, filename)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None

best_hybrid = load_best_params('best_params_hybrid.json')
best_classical = load_best_params('best_params_classical.json')

print(f'Parametros Hibridos: {best_hybrid if best_hybrid else "(Usando Backup)"}')
print(f'Parametros Classicos: {best_classical if best_classical else "(Usando Backup)"}')

## 3. Treinamento Final com Parametros Otimizados
Rodamos por 20 epocas para garantir a convergencia. O checkpoint salvara o melhor estado baseado na acuracia de validacao.

In [ ]:
# Hibrido
print('\n--- Treinando Modelo Hibrido Otimizado ---')
q_depth = best_hybrid['q_depth'] if best_hybrid else 1
lr_h = best_hybrid['lr'] if best_hybrid else 0.0004
wd_h = best_hybrid['weight_decay'] if best_hybrid else 1e-4

hybrid_model = HybridResNet18(num_classes=2, n_qubits=4, q_depth=q_depth)
hybrid_history = train_model(
    model=hybrid_model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    model_name="Hybrid_Notebook",
    epochs=20,
    lr=lr_h,
    weight_decay=wd_h,
    device=device,
    output_dir=output_notebook_dir
)

# Classico
print('\n--- Treinando Modelo Classico Otimizado ---')
lr_c = best_classical['lr'] if best_classical else 0.0004
wd_c = best_classical['weight_decay'] if best_classical else 1e-4

classical_model = ClassicalResNet18(num_classes=2)
classical_history = train_model(
    model=classical_model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    model_name="Classical_Notebook",
    epochs=20,
    lr=lr_c,
    weight_decay=wd_c,
    device=device,
    output_dir=output_notebook_dir
)

## 4. Inferencia em Teste Cego (Blind Test)
Avaliando a performance real em dados nunca vistos.

In [ ]:
print('\n' + '='*40)
print('RESULTADOS FINAIS NO TESTE CEGO')
print('='*40)

print('\n[HIBRIDO]')
acc_h, preds_h, true_h = test_model(hybrid_model, test_loader, device=device)

print('\n[CLASSICO]')
acc_c, preds_c, true_c = test_model(classical_model, test_loader, device=device)

print('\n' + '='*40)
print(f'Ganho Quantum: {((acc_h - acc_c) / acc_c * 100):.2f}%' if acc_c > 0 else 'N/A')
print('='*40)

# Salva resumo em txt
with open(os.path.join(output_notebook_dir, "notebook_summary.txt"), "w") as f:
    f.write(f"Acuracia H: {acc_h:.4f}\nAcuracia C: {acc_c:.4f}")

In [ ]:
print('--- Graficos de Treinamento ---')
plot_comparison(hybrid_history, classical_history, output_dir=output_notebook_dir)

print('\n--- Matrizes de Confusao ---')
plot_confusion_matrix(true_h, preds_h, "Hybrid", output_notebook_dir)
plot_confusion_matrix(true_c, preds_c, "Classical", output_notebook_dir)